#**Prueba de las funciones que hemos implementado**

## Instalación e importación de las librerías y bibliotecas necesarias para las funciones

In [1]:
# Primero instalamos las bibliotecas y/o librerías necesarias para llevar a cabo dicho preprocesamiento

!pip install wfdb

In [2]:
# Importamos las bibliotecas necesarias para poder trabajar con los distintos conjuntos de datos procedentes de PhysioNet

# Aunque ya se hayan cargado en el archivo de Python con las funciones lo volvemos a hacer por si la importación no ha sido correcta

import numpy as np
import os
import matplotlib.pyplot as plt
import wfdb
import sys


from scipy.signal import butter, filtfilt, find_peaks

# Carga del archivo .py con las funciones

In [3]:
# Nos montamos a Google Drive para poder cargar los archivos con las funciones necesarias

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Añadimos la ruta en la que se enceuntran estos archivos para poder acceder a ellos

sys.path.append('/content/drive/MyDrive/programacion_ECG')

In [5]:
# Importamos las funciones del archivo, para comprobar que funcionan correctamente

from funcionescargadatos import carga_archivos_entorno
from funcionespreprocesamientodatos import *
from funcionesrepresentacionsenales import *

# Función que carga el dataset con el que vamos a hacer las respectivas pruebas

## Carga del conjunto de datos de validación desde Drive en el entorno actual de trabajo

Se utiliza el conjunto de datos de validación para esta prueba puesto que su longitud es menor y será más rápido de procesar

In [6]:
# Inicializamos la ruta de Google Drive en la que se encuentra dicho conjunto de datos

ruta_datos_validacion = "/content/drive/MyDrive/dataset_ECG/challenge2017/validacion"


In [7]:
# Realizamos el proceso de carga de los registros wfdb.Record de estos datos

diccionario_señales_validacion = carga_archivos_entorno(ruta_datos_validacion)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Se ha cargado el archivo A00001 correctamente.
Se ha cargado el archivo A00039 correctamente.
Se ha cargado el archivo A00002 correctamente.
Se ha cargado el archivo A00040 correctamente.
Se ha cargado el archivo A00003 correctamente.
Se ha cargado el archivo A00041 correctamente.
Se ha cargado el archivo A00004 correctamente.
Se ha cargado el archivo A00042 correctamente.
Se ha cargado el archivo A00005 correctamente.
Se ha cargado el archivo A00043 correctamente.
Se ha cargado el archivo A00006 correctamente.
Se ha cargado el archivo A00044 correctamente.
Se ha cargado el archivo A00007 correctamente.
Se ha cargado el archivo A00045 correctamente.
Se ha cargado el archivo A00008 correctamente.
Se ha cargado el archivo A00046 correctamente.
Se ha cargado el archivo A00009 correctamente.
Se ha cargado el archivo A00047 correctamente.
Se ha cargado el archivo 

In [8]:
# Comprobamos que se han cargado correctamente estas señales
longitud = diccionario_señales_validacion.keys()
print (len(longitud))

300


# Función que obtiene la matriz de Numpy con los datos numéricos de la señal

Lo primero que vamos a hacer es extraer la señal monocanal o de una derivada que conforma nuestro ECG, ya que nuestro dataset se encuentra formado únicamente por señales de este tipo.

Esto es necesario porque el objeto wfdb.Record que tenemos presenta un montón de información acerca de esta señal, sin embargo, lo que a nosotros nos interesa para la red GAN que vamos a entrenar son los valores numéricos de la señal, la matriz de NumPy que los recoge.

Además, como la señal es unidimensional podemos trabajar con la señal aplanada, es decir, como un array 1D, lo cual facilita el trabajo.

## Obtención de los valores numéricos de la señal

In [9]:
# Se muestran por pantalla los valores numéricos de cada señal junto a la longitud de la misma

for nombre, record in diccionario_señales_validacion.items():
  señal = obtener_señal_ecg(record)
  print (señal)
  print (len(señal))

[-0.127 -0.162 -0.197 ... -0.018 -0.022 -0.021]
9000
[0.387 0.461 0.538 ... 0.077 0.039 0.   ]
9000
[0.128 0.157 0.189 ... 0.    0.001 0.002]
9000
[ 0.326  0.401  0.479 ... -0.01  -0.01  -0.011]
9000
[ 0.056  0.073  0.085 ... -0.064 -0.036 -0.02 ]
18000
[-0.933 -1.107 -1.282 ...  0.225  0.021  0.023]
9000
[0.519 0.619 0.723 ... 0.116 0.017 0.018]
9000
[-0.063 -0.085 -0.102 ...  0.184  0.125  0.071]
9000
[-0.188 -0.239 -0.274 ... -0.093 -0.057  0.   ]
18000
[ 0.657  0.79   0.931 ...  0.376  0.191 -0.031]
14936
[-0.266 -0.316 -0.367 ...  0.051  0.034  0.021]
9000
[ 0.703  0.721  0.738 ... -0.098 -0.025  0.105]
9000
[0.021 0.022 0.024 ... 0.236 0.174 0.084]
9000
[-0.119 -0.154 -0.191 ... -0.027 -0.018 -0.011]
9000
[-0.187 -0.236 -0.286 ... -0.098 -0.033  0.019]
18000
[-0.104 -0.121 -0.134 ... -0.022 -0.02  -0.019]
9000
[ 0.051  0.056  0.059 ... -0.08  -0.069 -0.062]
9000
[-0.041 -0.038 -0.036 ... -0.013 -0.028 -0.046]
9000
[-1.028 -1.225 -1.418 ...  0.01   0.009  0.009]
18000
[ 0.19   0.2

# Función que realiza el filtrado de la señal en un rango de frecuencias

A continuación vamos a aplicar un filtro pasabanda a la señal para quedarnos con los rangos de frecuencia que nos interesan. Esto es importante sobre todo en señales del cuerpo humano para evitar aquellas frecuencias que pueden pertenecer a otro tipo de sonidos que no son ECGs. De esta forma eliminaremos los sonidos de baja y alta frecuencia respectivamente, los cuales pueden influir negativamente en la interpretación o entrenamiento de la red GAN, limpiando así nuestra señal.

- Sonidos de baja frecuencia (por debajo de 0.5): Movimiento del paciente (baseline wander o “deriva de línea base”), interferencia del contacto de electrodos, fluctuaciones del sensor.

- Sonidos de alta frecuencia (por encima de 40): Actividad muscular (EMG), ruido eléctrico (como interferencia a 50/60 Hz), artefactos del entorno.

## Filtrado de la señal dentro de un rango de frecuencias

In [10]:
# Se recorren las distintas señales del diccionario y se realiza el filtrado individual de cada una de ellas

lista_señales_filtradas = []

for nombre, record in diccionario_señales_validacion.items():
  señal_filtrada = filtro_pasa_banda (record)
  lista_señales_filtradas.append(señal_filtrada)



In [11]:
# Se imprimen las señales filtradas y la longitud del diccionario que contiene estas señales, es decir, el número de señales filtradas

print (lista_señales_filtradas)
print (len(lista_señales_filtradas))

[array([-0.04491202, -0.08218223, -0.11576399, ...,  0.00510383,
        0.00373707,  0.00251376]), array([ 0.33945792,  0.42120458,  0.50037497, ...,  0.04552434,
        0.01271697, -0.02247207]), array([ 0.01563098,  0.05008244,  0.08213577, ..., -0.01849   ,
       -0.01786012, -0.01709946]), array([0.11631395, 0.19988029, 0.27595872, ..., 0.00407129, 0.00371677,
       0.00312655]), array([ 0.03464   ,  0.04833872,  0.06067635, ..., -0.05898456,
       -0.03857461, -0.01649463]), array([-0.14346725, -0.3386947 , -0.52472985, ...,  0.1734158 ,
        0.06015328, -0.021514  ]), array([ 0.12868731,  0.24185245,  0.34822328, ...,  0.08091853,
        0.01522161, -0.01399549]), array([-0.02248848, -0.04296478, -0.06305813, ...,  0.26389545,
        0.21102159,  0.15580401]), array([-0.05855155, -0.10600634, -0.15032134, ..., -0.05256609,
       -0.00117434,  0.05004227]), array([0.18461729, 0.33597414, 0.4777829 , ..., 0.42345088, 0.26520826,
       0.07330466]), array([-0.08086703, -

# Función que normaliza y escala la señal entre -1 y 1

Es necesario llevar a cabo un proceso de normalización de los segmentos obtenidos a partir de las señales del dataset por los siguientes motivos:

- Estabilidad y velocidad de entrenamiento
El escalado hace que los valores de entrada estén en un rango similar (por ejemplo, [−1,1]). Esto evita que la red tenga que aprender a lidiar con rangos de datos muy amplios o con valores muy grandes o muy pequeños, lo que ayuda a que el entrenamiento sea más estable y converja más rápido.

- Evitar sesgo por magnitudes diferentes
Sin normalización, segmentos con amplitudes más grandes podrían dominar el aprendizaje, y la red podría ignorar patrones importantes en señales con amplitudes menores.

- Mejor rendimiento en redes neuronales
Muchas funciones de activación (como tanh, sigmoid, ReLU) funcionan mejor cuando los datos de entrada están escalados adecuadamente.

- Facilita la comparación entre señales
Si diferentes segmentos vienen de distintos pacientes o condiciones, normalizarlos ayuda a que el modelo vea todos los datos "en igualdad de condiciones", centrando la atención en las formas y patrones, no en la escala absoluta.

- Consistencia con datos sintéticos
En GANs, cuando generas señales sintéticas, también quieres que estén en el mismo rango de valores que los datos reales para que el discriminador no distinga fácilmente señales por escala.



## Función que calcula la media y desviación típica para la globalidad del dataset

In [12]:
# Es recomendable utilizar una media y desviación típica común para todo el dataset

#Por ello, se calculan estos parámetros para la normalización del tipo Z - Score

media_dataset, desviacion_dataset = obtener_param_globales (diccionario_señales_validacion)

print (media_dataset)
print (desviacion_dataset)

0.005152370653346714
0.2988346249537129


## Función que realiza la normalización y escalado de las señales

In [13]:
# Se lleva a cabo la normalización de tipo Z - Score y el escalado individual para cada una de las señales del dataset

for i in lista_señales_filtradas:
  segm_norm_esc = normalizar_señal (i, media_dataset, desviacion_dataset)


In [14]:
# Representación de algunas de las señales normalizadas por pantalla

for i in lista_señales_filtradas:
  segm_norm_esc_rep = normalizar_señal (i, media_dataset, desviacion_dataset)
  representacion_señal_ecg (segm_norm_esc_rep)



Output hidden; open in https://colab.research.google.com to view.

# Función de segmentación de la señal en base al pico R del complejo QRS

Nuestros ECG son un tipo de datos fisiológicos. Cada una de las señales que aparecen en el TFG no es una señal aislada, sino que se trata de una grabación del ritmo cardíaco durante un periodo de tiempo determinado, aproximadamente 1 minuto.

Si, por ejemplo, nuestra señal tiene 18.000 muestras quiere decir que se han tomado los valores discretos de esa función en 18.000 puntos distintos del ECG.

Sin embargo, para entrenar una red GAN con datos de este tipo es mucho más recomendable hacerlo con datos de longitud pequeña y fija, es decir, por ejemplo de un total de 300 muestras o puntos. Ese es el motivo por el que vamos a realizar la segmentación de la señal en ventanas o segmentos de la misma.
Esto a su vez actúa como una especie de Data Augmentation, ya que vamos a tener una mucho mayor cantidad de señales que pasarle a la red GAN para su entrenamiento.

Esta segmentación la vamos a llevar a cabo centrando nuestra atención sobre el complejo QRS, es decir, la onda más característica del ciclo cardíaco. Para ello, vamos a tomar el pico R como referencia y vamos a tomar una serie de muestras o datos hacia los dos lados de este pico. Este número de muestras va a encontrarse en base 2 (2^n) y va a depender de los resultados que obtengamos a la hora de entrenar nuestra red GAN. De esta forma, podremos también captar dentro de cada una de las miniseñales las ondas P y T que siguen a ambos lados a este complejo.

Además, de esta forma no perdemos información acerca de la señal, lo cual pasa con ventanas de tamaño fijo, ya que puede que cortemos esta en medio de un pico. Es decir, de esta forma es mucho más fácil y habitual que se produzca una pérdida de información.


## Proceso de detección de los picos en la señal

In [15]:
# Representamos las señales con los picos que se han detectado en las mismas

for i in lista_señales_filtradas:
  señal_norm = normalizar_señal (i, media_dataset, desviacion_dataset)
  segmentos_qrs, picos = segmentacion_complejo_qrs(señal_norm, frec_muestreo=300, exponente=7)


No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.


In [22]:
# Representación por pantalla de los picos encontrados en la señal

for i in lista_señales_filtradas:
  señal_norm = normalizar_señal (i, media_dataset, desviacion_dataset)
  segmentos_qrs, picos = segmentacion_complejo_qrs(señal_norm, frec_muestreo=300, exponente=7)


  if segmentos_qrs is None or picos is None:
    print (f"Registro {nom} omitido por tratarse de una señal vacía.")
    continue

  else:
    representacion_picos_señales (i, picos)

Output hidden; open in https://colab.research.google.com to view.

## Proceso de segmentación de la señal

In [19]:
# Representación por pantalla de las distintas ventanas en que se segmenta la señal

for i in lista_señales_filtradas:
  señal_norm = normalizar_señal (i, media_dataset, desviacion_dataset)
  segmentos, indices = segmentacion_complejo_qrs (señal_norm)
  print(len(segmentos), indices)
  for j in segmentos[:2]:
    representacion_señal_ecg (j)



Output hidden; open in https://colab.research.google.com to view.

# Función de Pipeline para el procesamiento completo de los datos

Una vez que hemos definido de forma individual todas las funciones que van a realizar el procesamiento de nuestra señal, podemos crear un PipeLine en forma de función que realice todas estas pequeñas modificaciones necesarias y hacer así que los datos lleven a cabo el procesamiento en la totalidad de su conjunto, sin tener que realizar la llamada a cada una de las funciones individualmente.

## Función de procesamiento individual de los registros

In [23]:
for nom, record in diccionario_señales_validacion.items():
  segmentos_procesados = procesar_record_individual (record, media_dataset, desviacion_dataset)


  if segmentos_procesados is None:
    print (f"Registro {nom} omitido por tratarse de una señal vacía.")
    continue


No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.
No se han encontrado picos válidos en

In [24]:
# Representación por pantalla de algunos de los segmentos generados

for nom, record in diccionario_señales_validacion.items():
  segmentos_procesados = procesar_record_individual (record, media_dataset, desviacion_dataset)


  if segmentos_procesados is None:
    print (f"Registro {nom} omitido por tratarse de una señal vacía.")
    continue

  for idx, segmento in enumerate(segmentos_procesados):
    if idx < 3:
      representacion_señal_ecg (segmento)

Output hidden; open in https://colab.research.google.com to view.

## Función de preprocesamiento del dataset completo

In [ ]:
a = procesar_multiples_registros (diccionario_señales_validacion)

print (a[0][0])

No se pudo segmentar la señal: No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.. No obtenemos picos en la misma.
El registro A00042 no pudo ser procesado y fue omitido.
No se pudo segmentar la señal: No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.. No obtenemos picos en la misma.
El registro A00013 no pudo ser procesado y fue omitido.
No se pudo segmentar la señal: No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.. No obtenemos picos en la misma.
El registro A00087 no pudo ser procesado y fue omitido.
No se pudo segmentar la señal: No se han encontrado picos válidos en la señal. Esta es muy corta o no presenta picos. Inténtalo de nuevo.. No obtenemos picos en la misma.
El registro A00139 no pudo ser procesado y fue omitido.
No se pudo segmentar la señal: No se han encontrado picos válidos en la señal. Esta es muy c

# Función de descarga del dataset procesado

In [ ]:
# Definición de la ruta de guardado de las señales procesadas

ruta_almacenamiento = "/content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN"

# El nombre del archivo debe presentar la extensión '.npy' ya que se trata de la extensión específica para objetos NumPy (como el array 2D en este caso),
# lo cual es útil para hacer posteriormente la carga en la red GAN

In [ ]:
descargar_segmentos_procesados (a, ruta_almacenamiento, "descarga_prueba.npy")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Los segmentos procesados han sido guardados correctamente en la ruta /content/drive/MyDrive/dataset_ECG/challenge2017/entrenamiento_GAN/descarga_prueba.npy.
